<b><font size="6" color="#E8800A">Week 4b · Feature work on a regression target</font></b><br>
<b><font size="4">Selection on a continuous target, and one transform that backfires</font></b><br>

The cleaning recipe produces `cars4you.csv`: 4,000 used cars whose preparation is documented. A feature or transform is kept only if it lowers
the held-out error.

Engineer features before selection, then compare filters, wrappers and embedded
methods.

> **Error is reported in euros throughout.** Mean absolute error on a price is a
> quantity a car buyer or dealer can interpret directly; $R^2$ is not.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>The Pre-Processing Regime</font>](#regime)
* [<font color='#E8800A'>Encoding and scaling</font>](#encoding)
* [<font color='#E8800A'>Feature extraction and engineering</font>](#transform)
* [<font color='#E8800A'>A combination feature, and the transform that undoes it</font>](#combination)
* [<font color='#E8800A'>Dimensionality reduction</font>](#svd)
* [<font color='#E8800A'>Feature selection I: Filter methods</font>](#filter)
* [<font color='#E8800A'>Feature selection II: Wrapper methods</font>](#wrapper)
* [<font color='#E8800A'>Feature selection III: Embedded methods</font>](#embedded)
* [<font color='#E8800A'>Combining strategies</font>](#combined)
* [<font color='#E8800A'>So what is selection for?</font>](#synthesis)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>The Pre-Processing Regime</font> <a class="anchor" id="regime"></a>
[Back to TOC](#toc)

The cleaning log carries seven decisions, and this notebook takes three
of them: the column roles with their fill rules (the training median for a
numeric gap, a level of its own for a missing category), `log1p` for `mpg` and
`mileage`, and keeping outliers. It stores the rules, not fitted
values. Each split must learn its medians, encoder statistics and scaling limits
from training rows; fitting them before the split would leak held-out information.

This notebook adds the encoding and scaler chosen by a sixteen-regime benchmark.
Together, those entries define the preprocessing recipe used below.

Every result below uses one protocol. Twenty repeated 80/20 splits are drawn
at the start. Every learned step, from filling a gap to
choosing a feature, is fitted on a split's training 80% and scored on its
held-out 20%, and every comparison is made split by split.

__Step 1:__ Import the libraries and set up the notebook. The setup
installs `category_encoders` only when it is missing.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("category_encoders") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "category-encoders"]
    )

# `course_helpers.py` sits beside the notebook, and Python does not search
# that folder on its own.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Standard library: `Counter` tallies how often a column is selected,
# `inspect` reads a selector's signature, `textwrap` folds printed text.
import inspect
import textwrap
import warnings
from collections import Counter

# Arrays, frames and plots.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

# Encoders for the categorical columns, and the two association statistics the
# typed filter scores a source column with.
from category_encoders import CountEncoder, TargetEncoder
from scipy.stats import f_oneway, pearsonr

# Feature extraction first, then the selectors this week compares.
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_selection import (
    RFE,
    RFECV,
    SelectFromModel,
    SelectKBest,
    SequentialFeatureSelector,
    VarianceThreshold,
    f_regression,
)

# The models every technique is scored through, the splits they are scored on,
# and the two errors reported for them.
from sklearn.linear_model import Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import ShuffleSplit

# The scalers and encoders refitted inside every training split.
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)

# The course's shared cleaning log and plot colours.
from course_helpers import CleaningLog, PLOT_BLUE, PLOT_ORANGE

RANDOM_STATE = 42
FIGSIZE = (6, 5)

# One-hot and pandas deprecation warnings are silenced once here. The source of
# each is known and expected.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

__Step 2:__ Load the regression spine and separate the target from the
features. `price` is the target. `model` is dropped because 310 distinct values
would make it a separate high-cardinality encoding problem.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

In [ ]:
cars = pd.read_csv("../../data/interim/cars4you.csv")

X = cars.drop(columns=["price", "model"])
y = cars["price"]

# Week 3's repeated 80/20 design. The target is a price, so there is nothing to
# stratify on and each split is a plain random draw.
splitter = ShuffleSplit(n_splits=20, test_size=0.20, random_state=RANDOM_STATE)
splits = list(splitter.split(X))

print(f"{X.shape[0]:,} rows x {X.shape[1]} source columns")
print(f"{X.isna().sum().sum():,} gaps in the file,"
      f" filled per split from that split's own training rows")
print(f"price: median {y.median():,.2f} EUR, "
      f"{y.min():,} to {y.max():,}")
print(f"{len(splits)} random 80/20 splits,"
      f" {len(splits[0][1])} rows held out in each")

__Step 3:__ Open the cleaning log. Each `CleaningLog` entry records the
affected columns, the decision, its reason, the number of changed rows, and a
machine-readable `carries` value.

In [ ]:
recipe_log = CleaningLog.load("../../logs/cars4you_cleaning_log.json")

print(f"{len(recipe_log)} decisions inherited from Week 3\n")
for decision in recipe_log.steps:
    # `carries` is empty for a step whose consequence is already in the file.
    carried = ", ".join(sorted(decision.carries)) if decision.carries else "-"
    print(f"  {decision.column:28s} {decision.action}")
    print(f"  {'':28s} carries -> {carried}\n")

__Step 4:__ Read the column roles and fill rules from the log rather than
retyping them. `plan` finds a labelled step and returns its `carries`. Here,
numeric gaps take training medians while categorical gaps become their own
level instead of the training mode.

In [ ]:
fill_plan = recipe_log.plan("11 feature columns")
numeric = [column for column in fill_plan["numeric"] if column in X.columns]
categorical = [column for column in fill_plan["categorical"] if column in X.columns]
numerical_fill = fill_plan["fill"]["numeric"]
categorical_fill = fill_plan["fill"]["categorical"]

print(f"{len(numeric)} numeric, {len(categorical)} categorical {categorical}")
print(f"numeric gaps take the {numerical_fill} of the training rows")
print(f"categorical gaps take the {categorical_fill!r} treatment")

__Step 5:__ Read the logged `log1p` columns and declare the scaler candidates.
Encoding and scaling remain open until the benchmark measures them together.

In [ ]:
log1p_columns = recipe_log.plan("mpg, mileage")["columns"]
scaler_candidates = ("none", "standard", "min-max", "robust")

print(f"log1p on {log1p_columns};"
      f" the other {len(numeric) - len(log1p_columns)} numeric columns are"
      f" {[c for c in numeric if c not in log1p_columns]}")

<div class="alert alert-block alert-info">

**A stored rule is not a stored fit.** Each training split supplies
its own medians, encoder statistics and scaling limits; the explicit missing
level and `log1p` are stateless. The benchmark crosses every encoder with all
four scaler regimes.

</div>

__Step 6:__ Define the fold mechanics before reading any result. The three
functions fill gaps, encode categoricals and scale numerics. Their fit rows must
remain visible because every score depends on those boundaries.

In [ ]:
scaler_classes = {"none": None, "standard": StandardScaler,
                  "minmax": MinMaxScaler, "min-max": MinMaxScaler,
                  "robust": RobustScaler}


def fill_from_training_rows(X_train, X_test, numeric, categorical,
                            numerical_fill="median", categorical_fill="mode"):
    """Fill both halves of a fold from the TRAINING half's own statistics.

    One rule per block, both named by the Week 3 log: the numeric columns take a
    median or a mean, the categorical ones a mode or a level of their own.

    The spine leaves its gaps open on purpose, so each fold fills its own, and
    both halves take the SAME values, computed from the training rows alone.

    Week 3 chose `"own level"` for this dataset, which does not learn anything
    from the rows at all: a gap becomes the explicit level `"(missing)"` and the
    encoder treats it as one more category.
    """
    numeric, categorical = list(numeric), list(categorical)
    if numerical_fill == "median":
        fills = {column: X_train[column].median() for column in numeric}
    elif numerical_fill == "mean":
        fills = {column: X_train[column].mean() for column in numeric}
    else:
        raise ValueError(f"unknown numerical fill: {numerical_fill!r}")
    for column in categorical:
        if categorical_fill == "own level":
            fills[column] = "(missing)"
        elif categorical_fill == "mode":
            mode = X_train[column].mode(dropna=True)
            if len(mode):
                fills[column] = mode.iloc[0]
        else:
            raise ValueError(f"unknown categorical fill: {categorical_fill!r}")
    # Copies. A fold must not mutate the frame the next fold reads.
    return X_train.fillna(fills), X_test.fillna(fills)


# One encoder per candidate, refitted inside every split. One-hot always
# drops one level per column.
encoders = {
    "one-hot": OneHotEncoder(handle_unknown="ignore", drop="first",
                             sparse_output=False),
    "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value",
                              unknown_value=-1),
    "count": CountEncoder(normalize=True, handle_unknown=0,
                          handle_missing="value"),
    "target": TargetEncoder(handle_unknown="value", handle_missing="value"),
}


def fold_matrices(X_train, X_test, categorical, numeric, log1p_columns=(),
                  scaler="standard", encoding="one-hot", y_train=None,
                  numerical_fill="median", categorical_fill="mode"):
    """Return (A_train, A_test, feature_names) for ONE fold.

        Every `.fit` takes `X_train`, and `X_test` only ever reaches a `.transform`.

    `log1p_columns` names RAW numeric columns

    The categorical block comes first in the matrix and the numeric block
    second. That width is not constant across folds, because a level rare enough
    to be absent from a training fold contributes no column to that fold.
    """
    numeric, categorical = list(numeric), list(categorical)

    # 1. FILL first, because the spine ships with its gaps open.
    X_train, X_test = fill_from_training_rows(
        X_train, X_test, numeric, categorical,
        numerical_fill=numerical_fill,
        categorical_fill=categorical_fill,
    )

    # 2. ENCODE. Target encoding reads price, so it gets the training prices
    #    only.
    encoder = encoders[encoding]
    if encoding == "target":
        if y_train is None:
            raise ValueError("target encoding requires y_train")
        cat_train = encoder.fit_transform(X_train[categorical], y_train)
    else:
        cat_train = encoder.fit_transform(X_train[categorical])
    cat_test = encoder.transform(X_test[categorical])
    names = list(encoder.get_feature_names_out(categorical)) + numeric

    # 3. log1p. Stateless, but kept inside the fold so that the order
    #    fill -> encode -> log1p -> scale is the same everywhere.
    num_train = X_train[numeric].to_numpy(dtype=float).copy()
    num_test = X_test[numeric].to_numpy(dtype=float).copy()
    if len(log1p_columns):
        where = [numeric.index(column) for column in log1p_columns]
        num_train[:, where] = np.log1p(num_train[:, where])
        num_test[:, where] = np.log1p(num_test[:, where])

        # 4. SCALE. The centre and the spread come from the training rows.
    make = scaler_classes[scaler]
    if make is not None:
        fitted = make().fit(num_train)
        num_train = fitted.transform(num_train)
        num_test = fitted.transform(num_test)
    # Ordinal, count and target encoding turn a level into a NUMBER, so the
    # scaler under test applies to them too. One-hot indicators stay 0/1.
    if make is not None and encoding != "one-hot":
        fitted = make().fit(cat_train)
        cat_train = fitted.transform(cat_train)
        cat_test = fitted.transform(cat_test)

    return (np.hstack([cat_train, num_train]),
            np.hstack([cat_test, num_test]),
            np.asarray(names))

<div class="alert alert-block alert-info">

These helpers are written in the notebook rather than imported,
so every fit boundary stays visible at the point it matters. Read that as a
teaching arrangement and not as a recommendation: once the helpers multiply, a
notebook is the wrong home for them. From Week 5 onward this logic is
imported from `preprocessing.py` beside the notebook, and the split is worth
making in your own work: one module per job, preprocessing in one and
evaluation in another.

</div>

# <font color='#E8800A'>Encoding and scaling</font> <a class="anchor" id="encoding"></a>
[Back to TOC](#toc)

Three categorical columns have to become numeric ones. One-hot,
ordinal, count and target encoding make different assumptions and different
matrix widths.

<div class="alert alert-block alert-info">

**The candidates do not make the same claim.** One-hot keeps a
separate indicator for each non-reference level and always uses `drop="first"`.
Ordinal is compact but imposes an arbitrary numeric order. Count is also compact
and replaces a level with its frequency in the training split. Target encoding
uses price, so its fit receives only the training 80%; the held-out 20% never
contributes to its category means.

**A number an encoder made is still a number.** Ordinal, count and target
encoding each replace a level with a numeric score, and a penalised model reads
that score on whatever scale the encoder produced. So the scaler under test is
applied to those scores as well, fitted on the same training rows as the
measurements. One-hot indicators are left as 0/1: their spread only records how
common a level is.

</div>

__Step 7:__ Cross every encoder with every scaler: 320 fitted models, each with one diagnostics row. This cell
measures the candidates without choosing among them.

In [ ]:
encoding_candidates = ("one-hot", "ordinal", "count", "target")
encoding_rows = []

for encoding in encoding_candidates:
    for scaler_candidate in scaler_candidates:
        for split_number, (train_index, test_index) in enumerate(splits, 1):
            A_train, A_test, _ = fold_matrices(
                X.iloc[train_index], X.iloc[test_index],
                categorical, numeric, log1p_columns, scaler=scaler_candidate,
                encoding=encoding, y_train=y.iloc[train_index],
                numerical_fill=numerical_fill,
                categorical_fill=categorical_fill,
            )
            model = Ridge(random_state=RANDOM_STATE)
            model.fit(A_train, y.iloc[train_index])
            train_prediction = model.predict(A_train)
            test_prediction = model.predict(A_test)
            encoding_rows.append({
                "encoding": encoding,
                "scaler": scaler_candidate,
                "split": split_number,
                "train MAE": mean_absolute_error(y.iloc[train_index], train_prediction),
                "test MAE": mean_absolute_error(y.iloc[test_index], test_prediction),
                "train RMSE": root_mean_squared_error(y.iloc[train_index], train_prediction),
                "test RMSE": root_mean_squared_error(y.iloc[test_index], test_prediction),
            })

encoding_scores = pd.DataFrame(encoding_rows)
print(f"{len(encoding_scores)} fitted models:"
      f" {len(encoding_candidates)} encoders x {len(scaler_candidates)} scalers"
      f" x {len(splits)} splits")

__Step 8:__ Average each pair over the splits. Use held-out MAE to
choose, its standard error to judge the gap, and the training scores only to
diagnose fit.

In [ ]:
score_columns = ["train MAE", "test MAE", "train RMSE", "test RMSE"]
by_pair = encoding_scores.groupby(["encoding", "scaler"], sort=False)
encoding_board = by_pair[score_columns].mean()
# One standard error per mean: a training score is an estimate too.
encoding_board["train MAE SEM"] = by_pair["train MAE"].sem()
encoding_board["MAE SEM"] = by_pair["test MAE"].sem()

# idxmin, not idxmax: MAE is an error, so the smallest one wins. It returns the
# (encoding, scaler) label of the best row, not its score.
selected_encoding, selected_scaler = encoding_board["test MAE"].idxmin()
scaler = selected_scaler

print(encoding_board.round(2).to_string())
print(f"selected pair: {selected_encoding} + {selected_scaler}"
      " (lowest held-out MAE)")

# <font color='#E8800A'>Feature extraction and engineering</font> <a class="anchor" id="transform"></a>
[Back to TOC](#toc)

Feature engineering changes the representation before selection:
one column may be transformed, several may be combined, or a category may be
encoded. The cleaning recipe records `log1p` for `mpg` and `mileage`; this
section tests that choice and a case where applying it by habit fails.

__Step 9:__ Look at what `log1p` does to a column. Nothing here is fitted:
`log1p` is the same expression on every row, so it can be applied inside or
outside a split without changing a number or leaking one.

In [ ]:
# `.skew` reads the rows where the value is present and ignores the
# rest. This says what the transform does to the SHAPE of a column, and nothing
# about the price.
skew_table = pd.DataFrame({
    "min": [X[c].min() for c in numeric],
    "max": [X[c].max() for c in numeric],
    "skew": [X[c].skew() for c in numeric],
    "skew after log1p": [np.log1p(X[c]).skew() for c in numeric],
}, index=numeric).sort_values("skew", ascending=False)
print(skew_table.round(4).to_string())

column = "mpg"
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(X[column].dropna(), bins=40, color=PLOT_BLUE, ax=axes[0])
axes[0].set(title=f"{column}, as recorded", xlabel="miles per gallon")
sns.histplot(np.log1p(X[column].dropna()), bins=40, color=PLOT_BLUE, ax=axes[1])
axes[1].set(title=f"log1p({column})", xlabel="log1p(miles per gallon)")
fig.tight_layout()
plt.show()

`log1p` changes scale, not information. For `mpg`, it spreads the
dense part of the distribution and reduces skew from **3.0736** to **-0.5521**.

A long tail can pull a linear coefficient towards a few rows and dominate a
distance calculation. Compressing it restores resolution among typical rows,
but it cannot make an uninformative column useful and may make an informative
one worse.

<div class="alert alert-block alert-info">

Several alternatives cover different shapes:

- **`np.sqrt`** applies a milder, stateless compression to a right tail.
- [**`PowerTransformer`**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html)
  estimates a power. Yeo-Johnson accepts zero and negative values; Box-Cox
  requires strictly positive values.
- [**`QuantileTransformer`**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.QuantileTransformer.html)
  maps ranks to a chosen distribution, so distances change substantially.
- [**`KBinsDiscretizer`**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html)
  converts a measurement into ordered bands.

The last three differ from `log1p`: they learn powers, quantiles, or edges and
must be fit inside each training split. Apply them to raw numeric columns, not the encoded
matrix. **A 0/1 column cannot be de-skewed**: mapping it to 0 and 0.693 only
renames its two values.

</div>

`mpg` is the most skewed column at 3.07. `engineSize` is skewed too,
at 1.33, and a rule that logs every right-skewed column would take it. A column's
shape cannot tell you whether logging it helps the model, so the exercise below
measures it.

__Step 10:__ Two given helpers carry every comparison in the notebook.
`repeated_holdout_scores` splits, builds fold matrices, fits, and returns one
held-out score per split, and `paired_delta` reads two such arrays split by
split and returns the mean difference, the standard error of that mean, how many
splits the first array won, and the verdict at two standard errors. The selection
exercise later writes the loop explicitly.

In [ ]:
def repeated_holdout_scores(frame, y, categorical, numeric, log1p_columns=(),
                            *, splits, make_model, score, scaler="standard",
                            encoding="one-hot", numerical_fill="median",
                            categorical_fill="mode"):
    """One held-out score per split, for a frame and a preprocessing choice.

    The splits arrive already drawn, so every call scores on the same twenty
    held-out sets and the deltas between two calls pair up row by row. The
    preprocessing is rebuilt inside each split by `fold_matrices`, so nothing
    has seen the rows it is scored on.
    """
    scores = []
    for train_index, test_index in splits:
        A_train, A_test, _ = fold_matrices(
            frame.iloc[train_index], frame.iloc[test_index],
            categorical, numeric, log1p_columns, scaler=scaler,
            encoding=encoding, y_train=y.iloc[train_index],
            numerical_fill=numerical_fill,
            categorical_fill=categorical_fill,
        )
        model = make_model().fit(A_train, y.iloc[train_index])
        prediction = model.predict(A_test)
        scores.append(score(y.iloc[test_index], prediction))
    return np.asarray(scores)


def paired_delta(scores, reference):
    """Mean split-by-split difference from `reference`, its SEM, wins and verdict."""
    delta = scores - reference
    sem = delta.std(ddof=1) / np.sqrt(len(delta))
    # Two standard errors separate a difference from noise; a delta that is
    # zero on every split is no difference at all. The metric is an error, so a
    # negative delta is the improvement.
    verdict = ("identical on every split" if not delta.any()
               else "inside the noise" if abs(delta.mean()) <= 2 * sem
               else "better" if delta.mean() < 0 else "worse")
    return {"dMAE": delta.mean(), "SEM": sem,
            "better": f"{int((delta < 0).sum())}/{len(delta)}", "verdict": verdict}

__Step 11:__ `heldout_mae` scores one frame: it hands the frame and the columns to
log to `repeated_holdout_scores` above, with this notebook's ridge regression and
mean absolute error, and returns one held-out error per split. Each candidate
below is judged by a *paired model delta*, one column transformed at a time and
compared split by split. Negative is better, because the metric is an error.

In [ ]:
def heldout_mae(log_columns, frame=X, numeric=numeric, scaler=scaler):
    """Twenty held-out errors for `frame`, logging `log_columns` in each split."""
    categorical = [c for c in frame.columns if c not in numeric]
    return repeated_holdout_scores(
        frame, y, categorical, numeric, log_columns,
        scaler=scaler, splits=splits,
        make_model=lambda: Ridge(random_state=RANDOM_STATE),
        score=mean_absolute_error,
        encoding=selected_encoding,
        numerical_fill=numerical_fill,
        categorical_fill=categorical_fill,
    )


# The reference every arm below is paired against: nothing logged.
baseline = heldout_mae([])

**Exercise.** Score the arms. Every arm below is the same frame, `X` as
loaded, and differs only in **which columns the recipe logs**, so every one of
them is judged against the same reference: `baseline`, the run that logs nothing.

| arm | columns logged | compared with |
|---|---|---|
| `baseline` | none | (the reference) |
| one column at a time | `mpg`, then `mileage`, then `engineSize` | `baseline` |
| the ruled pair | `mpg` and `mileage` | `baseline` |
| a shape rule | every right-skewed column | `baseline` |
| a blanket rule | all seven numeric columns | `baseline` |

In [ ]:
# 1. one row per column, each paired against `baseline`
singles = pd.DataFrame({
    f"log1p({column}) only": {
        "logged": column,
        **...}  # <-- CODE HERE
    for column in ("mpg", "mileage", "engineSize")
}).T
print(f"baseline, no log1p at all: MAE {baseline.mean():,.2f}")
print(singles.to_string(formatters={"dMAE": "{:+.2f}".format,
                                    "SEM": "{:.2f}".format}))


__Step 12:__ Now the whole-list candidates: the ruled pair, the "log everything
right-skewed" rule, and the blanket rule the cleaning notebook measured
and rejected. Same reference, so these rows and the ones above can be read as one table.

In [ ]:
right_skewed = [c for c in numeric if X[c].skew() > 1]
rules = pd.DataFrame({
    "log1p(mpg, mileage) RULED": {
        "logged": "2 columns", **paired_delta(heldout_mae(log1p_columns), baseline)},
    f"the {len(right_skewed)} right-skewed columns": {
        "logged": f"{len(right_skewed)} columns",
        **paired_delta(heldout_mae(right_skewed), baseline)},
    "all 7 numeric columns": {
        "logged": "7 columns", **paired_delta(heldout_mae(numeric), baseline)},
}).T
print(rules.to_string(formatters={"dMAE": "{:+.2f}".format,
                                  "SEM": "{:.2f}".format}))

<div class="alert alert-block alert-success">

**One wrong column reverses the gain.**

| `log1p` applied to | paired Δ MAE | splits it wins |
|---|---|---|
| `mpg` alone | **-52.87 ± 4.90 EUR**, helps | 19 of 20 |
| `mileage` alone | -16.90 ± 8.46 EUR | 13 of 20 |
| **`mpg` + `mileage`** *(the recipe)* | **-59.53 ± 8.34 EUR**, helps | 20 of 20 |
| `engineSize` alone | **+98.43 ± 5.57 EUR**, **hurts** | 0 of 20 |
| the three right-skewed columns | **+38.24 ± 10.07 EUR**, hurts | 3 of 20 |
| all seven numeric *(a blanket rule)* | **+36.37 ± 10.04 EUR**, hurts | 4 of 20 |

`mpg` supplies most of the gain. The individual `mileage` effect remains
uncertain because its standard error is about half its estimate. The ruled pair
wins every split, although its advantage over `mpg` alone remains within the
spread.

Adding `engineSize` more than cancels that gain. **A blanket "log your skewed
features" makes this model worse than doing nothing at all.** The width of its range does not separate it
from `mpg`; the measurement does, and it **hurts** on all twenty splits.

</div>

__Step 13:__ Measure the **target's** skew separately. Transforming the target is
a different decision from transforming a feature.

In [ ]:
print(f"price skew {y.skew():.4f}"
      f"   ->  log {np.log(y).skew():.4f}   log1p {np.log1p(y).skew():.4f}")

<div class="alert alert-block alert-info">

**`price` has a skew of 2.5450, and logging it gives -0.1123.** This
notebook does **not** do that, because transforming the target changes what the
error means: a mean absolute error on `log(price)` is not in euros and cannot be
compared directly with the euro MAE used throughout this notebook.

</div>

# <font color='#E8800A'>A combination feature, and the transform that undoes it</font> <a class="anchor" id="combination"></a>
[Back to TOC](#toc)

A new feature can come from more than one column. `mileage` is total
distance and `year` is age; their ratio is how hard the car has been driven per
year, which is a different thing from either.

__Step 14:__ Derive `km_per_year`, check its correlation with the price against both
parents on one split's training rows, then test it: first in the untransformed
frame, then in the ruled one.

In [ ]:
X_km = X.assign(km_per_year=X["mileage"] / (2026 - X["year"]))
numeric_km = numeric + ["km_per_year"]

# Pearson is the right index for two measurements, and it reads the price, so
# it is computed on one split's training rows and nowhere else.
train_index, _ = splits[0]
for column in ("km_per_year", "mileage", "year"):
    r = X_km[column].iloc[train_index].corr(y.iloc[train_index])
    print(f"|r({column:12s}, price)| {abs(r):.4f}")

ruled = heldout_mae(log1p_columns)
km = pd.DataFrame({
    "+ km_per_year, no log1p": {
        "reference": "baseline",
        **paired_delta(heldout_mae([], X_km, numeric_km), baseline)},
    "ruled frame + km_per_year": {
        "reference": "ruled",
        **paired_delta(heldout_mae(log1p_columns, X_km, numeric_km), ruled)},
}).T
print(km.to_string(formatters={"dMAE": "{:+.2f}".format, "SEM": "{:.2f}".format}))

`km_per_year` gains 7.45 EUR alone and **32.74 EUR** after the
recipe's transforms. Its value therefore depends on the frame it joins.

The gain is not predicted by univariate correlation. `km_per_year` correlates
**0.3822** with price, below `mileage` at 0.4203 and `year` at 0.4934, yet it
improves the fitted model beyond either parent.

__Step 15:__ Apply `log1p` to the new column on top of the ruled frame, then
measure its paired effect.

In [ ]:
with_km = heldout_mae(log1p_columns, X_km, numeric_km)
logged = paired_delta(
    heldout_mae(log1p_columns + ["km_per_year"], X_km, numeric_km), with_km)
print(f"and now log1p(km_per_year) too, against the ruled frame + km_per_year:"
      f"\n    dMAE {logged['dMAE']:+.2f} +- {logged['SEM']:.2f} EUR,"
      f" better on {logged['better']} splits -> {logged['verdict'].upper()}")
print(f"\ncorr(log1p(km_per_year), log1p(mileage)) ="
      f" {np.log1p(X_km['km_per_year']).corr(np.log1p(X_km['mileage'])):.4f}")

<div class="alert alert-block alert-success">

Logging `km_per_year` costs **+22.31 EUR** and wins only 3 of 20
splits. After the transform, its correlation with `log1p(mileage)` is **0.9943**,
so it nearly duplicates a column already in the frame.

The same transform helps two columns and hurts two others.
Choose it per column through a paired model delta.

</div>

# <font color='#E8800A'>Dimensionality reduction</font> <a class="anchor" id="svd"></a>
[Back to TOC](#toc)

Dimensionality reduction builds fewer substitute columns from the
originals. Selection instead retains a subset of the named originals. The
difference matters for both supervision and interpretation.

<div class="alert alert-block alert-info">

The main reducers preserve different structures:

- [**`PCA`**](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
  keeps orthogonal directions with the greatest variance in a centred matrix.
- [**`TruncatedSVD`**](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)
  avoids centring and therefore preserves sparsity, which suits text matrices.
- [**`NMF`**](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)
  represents each row as additive non-negative parts.
- [**`FeatureAgglomeration`**](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.FeatureAgglomeration.html)
  averages groups of similar columns.
- [**Random projection**](https://scikit-learn.org/stable/modules/random_projection.html)
  uses a random matrix to approximately preserve distances in very wide data.
- [**`TSNE`**](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)
  and [UMAP](https://umap-learn.readthedocs.io/) provide nonlinear maps for
  two-dimensional inspection, not this predictive pipeline.

Each method is fitted and therefore belongs inside the training split.

</div>

__Step 16:__ See the idea on two columns before applying it to all of them.
`year` and `mileage` move together, because an older car has had longer to be
driven, so most of what they say lies along one direction.

In [ ]:
# One split's training rows, filled by the recipe's own rules: PCA is fitted, so
# it is fitted where every other fitted step in this notebook is. The frame goes
# in twice because the function fills a training half and a test half, and here
# both are the training rows.
train_index, _ = splits[0]
train_rows, _ = fill_from_training_rows(
    X.iloc[train_index], X.iloc[train_index], numeric, categorical,
    numerical_fill=numerical_fill, categorical_fill=categorical_fill,
)
train_y = y.iloc[train_index]

# `mileage` takes the recipe's log1p, and both are standardised, so the picture
# is about direction rather than units.
two = StandardScaler().fit_transform(np.column_stack([
    train_rows["year"].to_numpy(dtype=float),
    np.log1p(train_rows["mileage"].to_numpy(dtype=float)),
]))
pca_two = PCA(n_components=2).fit(two)
centre = two.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(two[:, 0], two[:, 1], s=8, alpha=0.25, color=PLOT_BLUE)
for spread, direction, name in zip(np.sqrt(pca_two.explained_variance_),
                                   pca_two.components_, ("PC1", "PC2")):
    tip = centre + 2 * spread * direction
    axes[0].annotate("", xy=tip, xytext=centre,
                     arrowprops=dict(arrowstyle="-|>", color=PLOT_ORANGE, lw=2))
    axes[0].text(*tip, f"  {name}", color="0.2", fontsize=11)
axes[0].set_aspect("equal")
axes[0].set(xlabel="year", ylabel="log1p(mileage)",
            title="Two columns, and the two directions PCA finds")

axes[1].hist(pca_two.transform(two)[:, 0], bins=40, color=PLOT_BLUE)
axes[1].set(xlabel="position along PC1", ylabel="cars",
            title="The same rows as one number along PC1")
fig.tight_layout()
plt.show()

print(f"share of the spread along PC1: {pca_two.explained_variance_ratio_[0]:.4f}")

PC1 points along the cloud and PC2 across it. Projecting each car
onto PC1 retains **78%** of the spread in one value. PCA repeats this search in
orthogonal directions across the full matrix.

__Step 17:__ Encode the training rows, with the numeric block standardised, and fit
`PCA` on them. `explained` holds the share of variance each component carries.

In [ ]:
encoder = OneHotEncoder(drop="first", sparse_output=False)
dummies = encoder.fit_transform(train_rows[categorical])
numbers = train_rows[numeric].to_numpy(dtype=float)
feature_names = np.asarray(
    list(encoder.get_feature_names_out(categorical)) + numeric
)
# The numeric block is standardised so that a mileage in kilometres does not own
# every component by its units alone.
frame = np.hstack([dummies, StandardScaler().fit_transform(numbers)])

pca = PCA().fit(frame)
explained = pca.explained_variance_ratio_
print(f"{frame.shape[1]} columns")

In [ ]:
# Nothing below is chosen from this chart. It shows how variance accumulates for
# both reducers, on one axis, because both curves are shares of the same total.
svd = TruncatedSVD(n_components=frame.shape[1] - 1,
                   random_state=RANDOM_STATE).fit(frame)
curves = pd.concat([
    pd.DataFrame({"components kept": np.arange(1, len(ratios) + 1),
                  "share of variance kept": ratios.cumsum(),
                  "reducer": name})
    for name, ratios in (("PCA", explained),
                         ("TruncatedSVD", svd.explained_variance_ratio_))
])

fig, ax = plt.subplots(figsize=FIGSIZE)
sns.lineplot(curves, x="components kept", y="share of variance kept",
             hue="reducer", palette=[PLOT_BLUE, PLOT_ORANGE],
             marker="o", markersize=3, ax=ax)
ax.axhline(0.80, color="grey", linewidth=1)
ax.set_ylim(0, 1.02)
ax.set(title="How variance accumulates, one component at a time")
ax.legend(title=None)
plt.show()

for name, ratios in (("PCA", explained),
                     ("TruncatedSVD", svd.explained_variance_ratio_)):
    reach = int(np.searchsorted(ratios.cumsum(), 0.80) + 1)
    print(f"{name:13s} first component {ratios[0]:.4f}"
          f"   components to reach 80%: {reach}")

__Step 18:__ **Exercise.** Report how many components reach 80%, 90% and 95% of the
variance, first from `explained`. Then read the same shares and the same three
counts out of `numpy.linalg.svd` on the centred matrix.

In [ ]:
# 1. the shares from the SVD of the CENTRED matrix
_, S, _ = ...  # <-- CODE HERE
singular = ...  # <-- CODE HERE

for label, shares in (("PCA", explained), ("SVD", singular)):
    # 2. the count of components whose running total reaches each level
    needed = [... for t in (0.80, 0.90, 0.95)]  # <-- CODE HERE
    print(f"{label}  first three shares {np.round(shares[:3], 4).tolist()}"
          f"   components for 80% / 90% / 95%: {needed}")


__Step 19:__ Read what the first component is made of.

In [ ]:
loadings = pd.Series(pca.components_[0], index=feature_names)
print(loadings.abs().sort_values(ascending=False).head(6).round(3).to_string())
print(f"\ncolumns with a loading above 0.05 in absolute value:"
      f" {int((loadings.abs() > 0.05).sum())} of {len(loadings)}")

**Seven** components carry 80% of the variance in these twenty-four
standardised columns, and the SVD of the centred matrix returns the same shares and the same
three counts, because that factorisation is what `PCA` runs. The first component loads
on **8** columns, led by `mileage`, `year`, and `tax`, so its effect cannot be
attributed to one named fact about the car.

__Step 20:__ Does the compressed frame predict as well as the full one? Inside
every training split, on the frame the recipe selected, keep the components that
reach 80% of that split's variance, each reducer counting for itself, then compare against all the features.

In [ ]:
make_reducer = {"PCA": lambda n: PCA(n_components=n),
                "TruncatedSVD": lambda n: TruncatedSVD(n_components=n,
                                                       random_state=RANDOM_STATE)}


def reduced_mae(reducer_name):
    scores, widths = [], []
    for train_index, test_index in splits:
        A_train, A_test, _ = fold_matrices(
            X.iloc[train_index], X.iloc[test_index],
            categorical, numeric, log1p_columns, scaler=scaler,
            encoding=selected_encoding, y_train=y.iloc[train_index],
            numerical_fill=numerical_fill,
            categorical_fill=categorical_fill,
        )
        if reducer_name is not None:
            # k is fitted too: the smallest number of components that keeps 80%
            # of this split's variance, by each reducer's own accounting.
            probe = make_reducer[reducer_name](A_train.shape[1] - 1).fit(A_train)
            k = int(np.searchsorted(probe.explained_variance_ratio_.cumsum(), 0.80) + 1)
            reducer = make_reducer[reducer_name](k)
            A_train = reducer.fit_transform(A_train)
            A_test = reducer.transform(A_test)
        widths.append(A_train.shape[1])
        model = Ridge(random_state=RANDOM_STATE).fit(A_train, y.iloc[train_index])
        scores.append(mean_absolute_error(y.iloc[test_index], model.predict(A_test)))
    return np.asarray(scores), widths


def mae_row(scores, widths, reference=None):
    # Width and error for one arm; a reduced arm also gets its paired gap.
    row = {"features": np.mean(widths), "MAE": scores.mean(),
           "MAE SEM": scores.std(ddof=1) / np.sqrt(len(scores))}
    return row if reference is None else {**row, **paired_delta(scores, reference)}


everything, widths = reduced_mae(None)
reduction = {"all features": mae_row(everything, widths)}
for reducer_name in ("PCA", "TruncatedSVD"):
    scores, widths = reduced_mae(reducer_name)
    reduction[reducer_name] = mae_row(scores, widths, everything)
reduction = pd.DataFrame(reduction).T
print(reduction.to_string(na_rep="", formatters={
    "features": "{:.1f}".format, "MAE": "{:,.2f}".format,
    "MAE SEM": "{:.2f}".format,
    "dMAE": lambda value: "" if pd.isna(value) else f"{value:+,.2f}",
    "SEM": lambda value: "" if pd.isna(value) else f"{value:.2f}",
}))

<div class="alert alert-block alert-warning">

Keeping 80% of each training split's variance requires **8** PCA
components or **9** TruncatedSVD components. The full width scores 3,087 EUR,
the 8 components score 5,366 and the 9 score 5,356, so MAE rises by **2,279.52
EUR** and **2,268.92 EUR**, respectively. Both lose on **all twenty** splits.

Variance is not supervised signal. A high-variance direction may say little
about price, while a predictive column may sit in a discarded component.
Reduction suits many redundant columns or visualization; this small frame of
named predictors instead calls for target-aware selection.

</div>

# <font color='#E8800A'>Feature selection I: Filter methods</font> <a class="anchor" id="filter"></a>
[Back to TOC](#toc)

Filters rank columns without fitting a predictive model. This
section introduces them one at a time, each with its implementation and its
result: a column that never varies, a column's **relevance** to the price, and a
column's **redundancy** with another column.

__Step 21:__ Every technique in this part is judged the same way, so the
machinery is written once here and each section below adds its own rows to one
shared `board` through three functions.

`run_selector` takes one technique and one split. It rebuilds the
preprocessing inside that split's training 80%, fits the technique there, fits a
`Ridge` regression on the columns the technique kept, and scores that model on
the split's held-out 20% as a mean absolute error in euros.

`measure` runs `run_selector` on every split and files the result in
`board` under the technique's name: a DataFrame with one row per split, holding
the number of columns kept, the error, and the kept columns. It also keeps the
selector every split fitted, so a later section can read what a technique decided across the splits rather than refit it.

`report` states the result first, in words: the technique's error against the
reference arm, the gap with its standard error, and whether that gap is better,
worse, or inside the split-to-split noise. The score is an error, so a negative
gap is the improvement. It then names the columns kept in most splits. The kept set is a **frequency** rather than one split's list, because
selection is refitted inside every split: a column kept in all twenty and a
column kept in three are different findings.

`report` also files its summary as one row of the `results` DataFrame, so
the techniques can be read side by side.

In [ ]:
def run_selector(make_selector, split_number):
    """One selector and one of the twenty splits, fitted on the training 80%."""
    train_index, test_index = splits[split_number]
    A_train, A_test, names = fold_matrices(
        X.iloc[train_index], X.iloc[test_index],
        categorical, numeric, log1p_columns, scaler=scaler,
        encoding=selected_encoding, y_train=y.iloc[train_index],
        numerical_fill=numerical_fill,
        categorical_fill=categorical_fill,
    )
    fitted_selector = None
    if make_selector is None:
        keep = np.ones(A_train.shape[1], dtype=bool)
    else:
        # The selector is handed the matrix WITH its column names, because a
        # filter that judges a categorical as a whole has to know which dummies
        # came from it.
        fitted_selector = make_selector(split_number).fit(
            pd.DataFrame(A_train, columns=names), y.iloc[train_index]
        )
        keep = fitted_selector.get_support()

    model = Ridge(random_state=RANDOM_STATE).fit(
        A_train[:, keep], y.iloc[train_index]
    )
    predicted = model.predict(A_test[:, keep])
    row = {"n": int(keep.sum()),
           "MAE": mean_absolute_error(y.iloc[test_index], predicted),
           "chosen": [names[keep]]}
    return row, fitted_selector


board, selector_examples, results = {}, {}, {}
fitted_by_split = {}
RESULT_FORMAT = {
    "columns": "{:.1f}".format,
    "MAE": "{:,.2f}".format,
    "dMAE": lambda value: "" if pd.isna(value) else f"{value:+.2f}",
    "SEM": lambda value: "" if pd.isna(value) else f"{value:.2f}",
}


def measure(name, make_selector):
    """Run one technique over every split, keeping each split's fitted selector."""
    rows, fitted_by_split[name] = [], []
    for split_number in range(len(splits)):
        row, fitted_selector = run_selector(make_selector, split_number)
        rows.append(row)
        fitted_by_split[name].append(fitted_selector)
        if split_number == 0 and fitted_selector is not None:
            selector_examples[name] = fitted_selector
    board[name] = pd.DataFrame(rows)
    return board[name]


def per_split(name, read):
    """`read` applied to every split's fitted selector, one row per split.

    Columns are matched by NAME: a level absent from a training split has no
    column there, so that split leaves it missing rather than zero.
    """
    return pd.DataFrame([pd.Series(read(fitted), index=fitted.feature_names_in_)
                         for fitted in fitted_by_split[name]])


def report(name):
    """Say what this technique scored against all features, and what it kept.

    The verdict comes first, in words. The kept set follows as a FREQUENCY over
    the splits, and only the columns kept in most of them are named.
    """
    table = board[name]
    kept = Counter(column for chosen in table["chosen"]
                   for names in chosen for column in names)
    row = {"columns": table["n"].mean(), "MAE": table["MAE"].mean()}
    reference = board["all features"]["MAE"]
    if ("evaluation" in table
            and table["evaluation"].eq("model-selection validation").all()):
        row["verdict"] = "no unbiased delta"
        line = (f"validation MAE {row['MAE']:,.2f} EUR with {row['columns']:.1f}"
                " columns; these splits chose this width, so no gap to all"
                " features is claimed")
    elif name == "all features":
        line = (f"MAE {row['MAE']:,.2f} EUR with {row['columns']:.1f} columns: the"
                " reference every other row is compared with")
    else:
        row.update(paired_delta(table["MAE"].to_numpy(), reference.to_numpy()))
        line = (f"MAE {row['MAE']:,.2f} EUR with {row['columns']:.1f} columns, against"
                f" {reference.mean():,.2f} with all features: dMAE {row['dMAE']:+.2f}"
                f" +- {row['SEM']:.2f} EUR, better on {row['better']} splits"
                f" -> {row['verdict'].upper()}")
    results[name] = row
    usual = [f"{c} {n}/{len(splits)}"
             for c, n in kept.most_common() if n > len(splits) / 2]
    rare = sum(n <= len(splits) / 2 for n in kept.values())
    print(name)
    print(textwrap.fill(line, width=84, initial_indent="    ",
                        subsequent_indent="    "))
    print(textwrap.fill("kept in most splits: " + ", ".join(usual), width=84,
                        initial_indent="    ", subsequent_indent="      "))
    if rare:
        print(f"    kept in half the splits or fewer: {rare} more"
              f" column{'' if rare == 1 else 's'}")

__Step 22:__ Start with the reference every filter is compared with: the
recipe's full encoded matrix, with no column removed.

In [ ]:
measure("all features", None)
report("all features")

Every row below is read against these two numbers. The
reference keeps 24.6 columns on average rather than a fixed 24, because
`transmission_Other` holds one car and `fuelType_Electric` holds one: a level
can be missing from a training split, and one-hot encoding fitted inside that
split then creates no column for it. Its error of 3,087.00 EUR is what each selector is compared with.

<div class="alert alert-block alert-info">

**`VarianceThreshold` measures spread, not relevance.** A variance of 0
means the column is constant and can be dropped.

A column with little spread may be uninformative, but variances are
comparable only when their units are. You can set an arbitrary threshold
(e.g. 0.01) to drop columns that vary less than that.

</div>

__Step 23:__ **`VarianceThreshold`.** Look at the spread it would read, on split
0's training matrix exactly as the recipe builds it: the one-hot block, the
measured block, and how many columns have no spread at all.

In [ ]:
first_split = 0
first_index, _ = splits[first_split]
first_matrix, _, first_names = fold_matrices(
    X.iloc[first_index], X.iloc[first_index], categorical, numeric,
    log1p_columns, scaler=scaler, encoding=selected_encoding,
    y_train=y.iloc[first_index], numerical_fill=numerical_fill,
    categorical_fill=categorical_fill,
)
variances = pd.Series(first_matrix.var(axis=0), index=first_names)
measured = variances.index.isin(numeric)
print(f"{len(variances)} columns, {int((variances == 0).sum())} constant")
print(f"dummies : variance {variances[~measured].min():.4f} to {variances[~measured].max():.4f}")
print(f"measured: variance {variances[measured].min():.4f} to {variances[measured].max():.4f}")

Nothing is constant, so a threshold of zero has nothing
to remove. The recipe scales the measured columns to [0, 1], which is what makes
the two ranges comparable at all, and they are still not measuring the same
thing: a dummy's variance is the prevalence times one minus the prevalence, so
it reports how common a level is, while a measured column's variance reports how
its values spread inside the range the training rows happened to cover. The
dummies here run from 0.0003 to 0.2486 and the measurements from 0.0049 to
0.1223, so a ranking by spread would put common levels above every measurement
in the frame.

In [ ]:
measure("VarianceThreshold(0)", lambda split_number: VarianceThreshold(0.0))
report("VarianceThreshold(0)")

The filter removed nothing, so its row is the reference
arm under a second name and the difference is exactly zero on all twenty splits.
`VarianceThreshold` is a constant-column detector, and this recipe produces no
constant columns. It is useful after a hard filter or a rare-level merge, where a
column can end up constant, and not as a way of ranking.

<div class="alert alert-block alert-info">

**Correlation has two uses in selection.** **Relevance** compares a
column with the price: does it carry information about what a car sells for?
**Redundancy** compares two columns with each other: do they record the same fact
twice? A filter needs both, because a ranking by relevance alone cannot see that
two of its top columns are copies of one another.

**The index must match the pair of types.** An encoded dummy asks whether one
level, such as Toyota, relates to price. The source column asks whether `Brand`
matters as a whole.

- For a **measurement against price**, use Pearson's r. The same arithmetic
  between a 0/1 dummy and the price is the point-biserial coefficient.
- For a **categorical against price**, use
  [one-way ANOVA](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html)
  and report the **correlation ratio η**, the share of price variation between
  levels, on the same 0-to-1 scale as |r|.

Encoding changes storage, not variable type, so the encoded filter below reads
each column with the index its own type allows, and
`SelectKBest(f_regression)` would rank the same columns in the same order,
because the F-statistic is a monotone function of that same |r|.

</div>

__Step 24:__ **Relevance, source columns.** Score every source column against the
price with the index its type allows, on split 0's training rows. An ANOVA also
depends on groups large enough to have a trustworthy mean, so print the smallest
one beside it.

In [ ]:
def anova_eta(column, outcome):
    """One-way ANOVA of the price across a categorical's levels, as an effect size.

    Returns the statistic, its p-value, the correlation ratio eta, and the
    smallest group. F answers "could these level means differ by chance"; eta
    answers "how much of the price's variation sits between the levels", which
    is what a filter is asking, and it lands on [0, 1] beside an |r|. A level
    holding a single car has a mean but no spread, so the p-value beside it is
    not one to lean on.
    """
    values = outcome.to_numpy(dtype=float)
    groups = [values[column.to_numpy() == level] for level in column.unique()]
    if len(groups) < 2:
        # One level present in these rows: no between-group variation exists to
        # compare against the within-group spread.
        return 0.0, float("nan"), 0.0, len(values)
    statistic, p = f_oneway(*groups)
    grand = values.mean()
    between = sum(len(group) * (group.mean() - grand) ** 2 for group in groups)
    eta = np.sqrt(between / ((values - grand) ** 2).sum())
    return statistic, p, eta, min(len(group) for group in groups)


def typed_relevance(rows, outcome):
    """One comparable number per SOURCE column, each by the index its type allows.

    |Pearson r| for a measurement, the correlation ratio eta for a categorical.
    Both are on [0, 1], so a ten-level column and a mileage can be ranked
    together. A measurement is read as the recipe leaves it, because that is the
    column the model will see.
    """
    scores = {}
    price = outcome.to_numpy(dtype=float)
    for column in numeric:
        values = rows[column].to_numpy(dtype=float)
        if column in log1p_columns:
            values = np.log1p(values)
        scores[column] = abs(pearsonr(values, price).statistic)
    for column in categorical:
        scores[column] = anova_eta(rows[column].astype(str), outcome)[2]
    return pd.Series(scores).sort_values(ascending=False)

In [ ]:
ranked = typed_relevance(train_rows, train_y)
table = []
for column in ranked.index:
    if column in numeric:
        table.append({"column": column, "index": "|Pearson r|",
                      "value": ranked[column]})
    else:
        statistic, p, eta, smallest = anova_eta(
            train_rows[column].astype(str), train_y)
        table.append({"column": column, "index": "eta (ANOVA)", "value": eta,
                      "levels": train_rows[column].nunique(), "F": statistic,
                      "p": p, "smallest group": smallest})
print(pd.DataFrame(table).round(4).to_string(index=False, na_rep=""))

<div class="alert alert-block alert-success">

`engineSize` leads at **0.6351**, followed by `transmission` at
**η 0.5528** and `Brand` at **0.5289**. The source-column calculation prevents those two
categoricals from appearing as thirteen unrelated dummies. The comparison is
valid because both η and |r| are bounded by 1.

`transmission` has **F = 351.57**, but one level contains **one car**. That group
has a mean but no estimated spread, so its p-value is not evidence. Report η as
the effect size and disclose the group count.

</div>

__Step 25:__ **`SelectKBest`, encoded columns.** It ranks the encoded columns one
at a time and keeps the top `k`. Its score function reads each column with the
index its own type allows, so a dummy is scored as a dummy and a measurement as a
measurement, and both land on [0, 1].

In [ ]:
def encoded_effect_scores(matrix, outcome):
    """One comparable [0, 1] effect for each ENCODED column against the price.

    |Pearson r| for a measured column, and the same arithmetic on a 0/1 dummy,
    where it is named the point-biserial coefficient. Naming the case is what
    keeps the reading inside what the types allow: a dummy's score is about one
    level, not about the categorical it came from.
    """
    price = np.asarray(outcome, dtype=float)
    matrix = np.asarray(matrix, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        scores = np.abs([pearsonr(matrix[:, j], price).statistic
                         for j in range(matrix.shape[1])])
    # A constant column correlates with nothing, and the arithmetic divides by
    # its zero spread.
    return np.nan_to_num(scores)

In [ ]:
# The claim in the box above, checked rather than asserted: the
# F-statistic is a monotone function of the same |r|, so it orders these columns
# exactly as the effect scores do.
print("f_regression gives the same ranking:",
      bool((np.argsort(f_regression(first_matrix, train_y)[0])
            == np.argsort(encoded_effect_scores(first_matrix, train_y))).all()))

measure("SelectKBest(k=10)",
        lambda split_number: SelectKBest(encoded_effect_scores, k=10))
report("SelectKBest(k=10)")

measure("SelectKBest(k=15)",
        lambda split_number: SelectKBest(encoded_effect_scores, k=15))
report("SelectKBest(k=15)")

Both budgets lose, and they lose on every split. Ten columns cost 169.58 EUR and fifteen cost 67.19, so the columns ranked
eleventh to fifteenth are worth about 100 EUR of error between them.

The kept sets show what a univariate ranking does with a ten-level categorical.
At k=10 the filter keeps `Brand_Mercedes` and `Brand_Opel` and drops the other
seven brand dummies, because it scores each level on its own and most levels are
individually weak. `Brand` as a variable is the third strongest column in the
table above.

__Step 26:__ **`TypedFilter`, source columns.** It asks the relevance question of
the source columns in the table above, keeps the `k` strongest, and retains every
dummy that came from a kept column.

In [ ]:
class TypedFilter:
    """Keep the k SOURCE columns with the strongest typed relevance.

    A dummy is not a column: ten of them can be one categorical. This filter
    scores `Brand` once, from an ANOVA across its levels, and keeps or drops all
    of its dummies together. It reads the split's own training rows, so nothing
    held out reaches the ranking.
    """

    def __init__(self, k, split_number):
        self.k = k
        self.split_number = split_number

    def fit(self, frame, outcome):
        train_index, _ = splits[self.split_number]
        rows, _ = fill_from_training_rows(
            X.iloc[train_index], X.iloc[train_index], numeric, categorical,
            numerical_fill=numerical_fill, categorical_fill=categorical_fill,
        )
        best = set(typed_relevance(rows, outcome).head(self.k).index)
        source_of = [next((c for c in categorical if name.startswith(c + "_")), name)
                     for name in frame.columns]
        self.support_ = np.array([source in best for source in source_of])
        return self

    def get_support(self):
        return self.support_

In [ ]:
measure("ANOVA / |r| on 5 source columns",
        lambda split_number: TypedFilter(5, split_number))
report("ANOVA / |r| on 5 source columns")

Five source columns expand to 15.8 encoded ones, because `Brand`
is kept or dropped as a unit and brings nine dummies with it. The cost is 165.04
EUR, about what the ten-column filter pays, and the kept set barely moves: `mileage` is kept in 19 splits and `mpg` in 1.
`mpg` ranks sixth in the typed table, just outside a budget of five, so it enters
only in the split whose ranking lifts it.

__Step 27:__ **Redundancy.** A relevance ranking compares each column only with
the price, so it cannot see two columns that record the same fact. Measure the
encoded columns against **each other**, on the same split-0 matrix, and name the
index each pair is entitled to.

In [ ]:
REDUNDANT = 0.70   # above this, treat a pair as one fact recorded twice


def pair_index(left, right):
    """Name the index a pair of columns is entitled to.

        All three are the same arithmetic on the same two vectors, and they carry
    different names because the types differ: Pearson's r between two
    measurements, the point-biserial coefficient between a measurement and a 0/1
    dummy, phi between two dummies.
    """
    measurements = sum(name in numeric for name in (left, right))
    return {2: "Pearson r", 1: "point-biserial", 0: "phi"}[measurements]

In [ ]:
between = np.abs(np.corrcoef(first_matrix, rowvar=False))
pairs_above = [
    (first_names[i], first_names[j], between[i, j])
    for i, j in zip(*np.triu_indices_from(between, 1))
    if between[i, j] > REDUNDANT
]
print(f"pairs above {REDUNDANT}: {len(pairs_above)} of"
      f" {between.shape[0] * (between.shape[0] - 1) // 2}")
for left, right, value in sorted(pairs_above, key=lambda row: -row[2]):
    print(f"  {left:24s} {right:24s} {value:.4f}  {pair_index(left, right)}")

One pair out of 276 crosses 0.70. `fuelType_Diesel` and
`fuelType_Petrol` reach phi 0.9028 because the remaining fuel types are rare, so
knowing a car is not diesel is almost the same fact as knowing it is petrol.
Mutually exclusive dummies from one source column are negatively related by
construction, and the size of that relation is a fact about one-hot encoding
rather than about fuel.

This frame holds little redundancy, so a rule that drops repeats has almost
nothing to act on.

__Step 28:__ **`CorrelationFilter`, relevance and redundancy together.** It walks
the same relevance ranking as `SelectKBest` and skips a candidate that repeats a
column already kept, so the budget is spent on distinct facts.

In [ ]:
class CorrelationFilter:
    """Rank by |r| with the price, then walk the ranking and skip repeats.

    A candidate is skipped when it correlates above `redundancy` with a column
    already kept: the pair is one fact recorded twice, and the second copy costs
    a slot the filter could have spent elsewhere. With `redundancy=None` this is
    the encoded `SelectKBest` baseline, and with `k=None` there is no budget to
    exhaust, so the redundancy rule runs alone and every column that repeats
    nothing already kept survives.
    """

    def __init__(self, k, redundancy=None):
        self.k = k
        self.redundancy = redundancy

    def fit(self, frame, outcome):
        matrix = frame.to_numpy(dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            relevance = encoded_effect_scores(matrix, outcome)
            between = np.nan_to_num(np.abs(np.corrcoef(matrix, rowvar=False)))
        keep = []
        for candidate in np.argsort(relevance)[::-1]:
            if len(keep) == self.k:
                break
            if self.redundancy is not None and any(
                between[candidate, chosen] > self.redundancy for chosen in keep
            ):
                continue
            keep.append(int(candidate))
        self.support_ = np.zeros(matrix.shape[1], dtype=bool)
        self.support_[keep] = True
        return self

    def get_support(self):
        return self.support_

In [ ]:
measure("correlation k=10, redundancy dropped",
        lambda split_number: CorrelationFilter(10, redundancy=REDUNDANT))
report("correlation k=10, redundancy dropped")

# Same budget, same ranking, one skip: the paired difference IS the rule.
skip = paired_delta(board["correlation k=10, redundancy dropped"]["MAE"].to_numpy(),
                    board["SelectKBest(k=10)"]["MAE"].to_numpy())
print(f"redundancy rule vs SelectKBest(k=10): dMAE {skip['dMAE']:+.2f}"
      f" +- {skip['SEM']:.2f}, better on {skip['better']} splits"
      f" -> {skip['verdict'].upper()}")

The redundancy rule changes nothing at all. Its row
is identical to `SelectKBest(k=10)` on every split, a paired difference of
exactly 0.00, because only `fuelType_Petrol` of the redundant pair reaches the
top ten and the skip is therefore never triggered.

__Step 29:__ Read the filters side by side. Every row is compared with `all features`.

In [ ]:
print(pd.DataFrame(results).T.to_string(formatters=RESULT_FORMAT, na_rep=""))

<div class="alert alert-block alert-success">

Every filter row that removes a column loses to the full matrix, and
the size of the loss tracks the width. `VarianceThreshold` removes nothing and matches the
reference exactly, fifteen columns give up 67.19 EUR, and the three rows at ten
to sixteen columns give up between 165 and 170.

The typed filter needs five quantities per car instead of ten, at a cost of
165.04 EUR on a 3,087 EUR error, about 5%.

</div>

# <font color='#E8800A'>Feature selection II: Wrapper methods</font> <a class="anchor" id="wrapper"></a>
[Back to TOC](#toc)

A wrapper judges features through a fitted model, so it can detect
joint effects that a univariate filter misses. Some wrappers cross-validate
internally, which adds another fit boundary.

__Step 30:__ **Exercise.** Inspect the three wrapper selectors' signatures for a
`cv` parameter. Classify which selectors run an internal validation loop and
which one applies a fixed feature budget without one.

In [ ]:
# 1. report each selector's cv default, or that it has none: a selector with
#    no cv parameter applies the feature budget you hand it
for cls in (RFE, RFECV, SequentialFeatureSelector):
    parameters = ...  # <-- CODE HERE
    if "cv" in parameters:
        # 2. `cv=None` is not "no cross-validation": the docstring says what the
        #    class does with it
        default = ...  # <-- CODE HERE
        folds = ...  # <-- CODE HERE
        print(f"{cls.__name__:26s} cv default = {default!r}, so {folds}-fold")
    else:
        print(f"{cls.__name__:26s} NO cv PARAMETER: fixed-budget fit")


<div class="alert alert-block alert-warning">

**Two wrappers cross-validate internally.** `RFECV` defaults to
five folds through `cv=None`, and `SequentialFeatureSelector` defaults to
`cv=5`. Preprocessing the outer training rows once before either selector leaks
information from each internal validation fold into its internal training fold.

The safe remedy is to fit preprocessing separately on each internal training
fold, transform its validation fold, and use an explicit `PredefinedSplit` to
keep those roles visible. The full implementation is deferred until nested
validation is introduced. This notebook stops at the warning and benchmarks
`RFE`, which has no internal cross-validation and takes its feature budget as an
argument, so the width is chosen outside the selector.

</div>

__Step 31:__ `RFE` fits the model, drops the column its coefficients rank last,
and refits until the budget is met. Start at ten columns, the budget the filters
above were given, so the two families are read at the same width.

In [ ]:
measure("RFE(k=10)", lambda split_number: RFE(
    Ridge(random_state=RANDOM_STATE), n_features_to_select=10))
report("RFE(k=10)")

Ten columns cost **+114.18** EUR here against **+169.58** for the
univariate filter at the same budget: the wrapper's fitted model ranks a column
against the columns already kept, which a column-by-column score cannot do.

Both numbers are penalties, so neither budget is worth paying. Ten is a width
this notebook picked, not one the data chose.

__Step 32:__ Choosing a width needs two helpers. `cache_rfe_folds` fits
preprocessing once per existing split and keeps that split's training and
validation matrices, so the k loop below never preprocesses anything.
`evaluate_rfe_k` fits one budget on every cached split and returns a row per
split, labelled as model-selection validation rather than as a held-out
score.

In [ ]:
def cache_rfe_folds():
    cached = []
    for split_number, (train_index, validation_index) in enumerate(splits):
        A_train, A_validation, names = fold_matrices(
            X.iloc[train_index], X.iloc[validation_index],
            categorical, numeric, log1p_columns, scaler=scaler,
            encoding=selected_encoding, y_train=y.iloc[train_index],
            numerical_fill=numerical_fill,
            categorical_fill=categorical_fill,
        )
        cached.append({
            "split": split_number,
            "train index": train_index,
            "validation index": validation_index,
            "train": A_train,
            "validation": A_validation,
            "names": names,
        })
    return cached


def evaluate_rfe_k(k, fold_cache):
    rows, fitted_selectors = [], []
    for fold in fold_cache:
        train_frame = pd.DataFrame(fold["train"], columns=fold["names"])
        validation_frame = pd.DataFrame(
            fold["validation"], columns=fold["names"]
        )
        train_y = y.iloc[fold["train index"]]
        validation_y = y.iloc[fold["validation index"]]
        selector = RFE(
            Ridge(random_state=RANDOM_STATE), n_features_to_select=k,
        ).fit(train_frame, train_y)
        keep = selector.get_support()
        rows.append({
            "n": int(keep.sum()),
            "train MAE": mean_absolute_error(
                train_y, selector.predict(train_frame)),
            "MAE": mean_absolute_error(
                validation_y, selector.predict(validation_frame)),
            "chosen": [fold["names"][keep]],
            "evaluation": "model-selection validation",
        })
        fitted_selectors.append(selector)
    return rows, fitted_selectors

__Step 33:__ Search the existing twenty train/validation splits: evaluate 5, 10,
15 and so on up to the common available width on their cached matrices. The
single width with the lowest mean validation error wins, and an exact tie prefers
the smaller model.

That error is **model-selection validation**, not an untouched test estimate,
because the same validation rows both choose `k` and describe it. Unbiased
nested evaluation is deferred until the course introduces that boundary.

In [ ]:
rfe_fold_cache = cache_rfe_folds()
common_width = min(fold["train"].shape[1] for fold in rfe_fold_cache)
candidate_counts = list(range(5, common_width + 1, 5))

rfe_candidates, curve_rows = {}, []
for k in candidate_counts:
    candidate = evaluate_rfe_k(k, rfe_fold_cache)
    rfe_candidates[k] = candidate
    fold_train_mae = np.asarray([row["train MAE"] for row in candidate[0]])
    fold_mae = np.asarray([row["MAE"] for row in candidate[0]])
    curve_rows.append({
        "k": k,
        "mean train MAE": fold_train_mae.mean(),
        "train MAE SEM": fold_train_mae.std(ddof=1) / np.sqrt(len(fold_train_mae)),
        "mean validation MAE": fold_mae.mean(),
        "validation MAE SEM": fold_mae.std(ddof=1) / np.sqrt(len(fold_mae)),
    })

rfe_curve = pd.DataFrame(curve_rows)
best_mean = rfe_curve["mean validation MAE"].min()
chosen_k = int(rfe_curve.loc[
    rfe_curve["mean validation MAE"] == best_mean, "k"
].min())
rfe_curve["selected"] = rfe_curve["k"].eq(chosen_k)

print(rfe_curve.round(2).to_string(index=False))
print(f"selected k = {chosen_k} by mean validation MAE; exact ties prefer smaller k")

The k=10 row of this table is the **3,201.17** EUR the board already
holds for `RFE(k=10)`, reached here by the same fits. Five columns are far too
few at 3,677.86, fifteen recover almost everything at 3,091.92, and twenty lands
lowest at 3,084.96.

The last two widths are **6.96** EUR apart with a standard error near 28, so they
are the same result measured twice; the rule takes the lowest mean anyway, and
that is the part a nested design fixes. Training error sits within nine euros of
validation error at every width, so nothing in this range is memorising its
rows.

Only the chosen width goes on the board, under one name.

In [ ]:
name = "RFE(validation-selected)"
chosen_rows, chosen_selectors = rfe_candidates[chosen_k]
board[name] = pd.DataFrame(chosen_rows)
fitted_by_split[name] = chosen_selectors
selector_examples[name] = chosen_selectors[0]

report("RFE(validation-selected)")

__Step 34:__ Plot the search. Both lines fall as columns are added, and the
marked point is the width the validation line landed on.

In [ ]:
selected_point = rfe_curve.loc[rfe_curve["selected"]].iloc[0]
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.errorbar(
    rfe_curve["k"], rfe_curve["mean train MAE"],
    yerr=rfe_curve["train MAE SEM"], marker="o", capsize=3,
    color=PLOT_BLUE, label="training",
)
ax.errorbar(
    rfe_curve["k"], rfe_curve["mean validation MAE"],
    yerr=rfe_curve["validation MAE SEM"], marker="o", capsize=3,
    color=PLOT_ORANGE, label="validation",
)
ax.axvline(chosen_k, color="0.45", linestyle=":", linewidth=1.5)
ax.scatter(
    [chosen_k], [selected_point["mean validation MAE"]],
    marker="*", s=180, color=PLOT_ORANGE, edgecolor="black", zorder=5,
)
ax.annotate(
    f"selected k={chosen_k}, validation MAE={selected_point['mean validation MAE']:,.2f}",
    xy=(chosen_k, selected_point["mean validation MAE"]),
    xytext=(8, 10), textcoords="offset points",
)
ax.set_xlabel("number of selected features")
ax.set_ylabel("mean absolute error (EUR)")
ax.set_title("RFE training/validation selection curve")
ax.legend()
fig.tight_layout()
plt.show()

`RFE` at that width keeps **20.0** of the 24.6 available columns at a
validation error of **3,084.96** EUR. The board carries no gap for this row: the
same twenty splits chose the width and then scored it, so the number is the
lowest of four candidates and optimistic by construction.

`RFE` supplies the ranking and those splits choose the width, so the board holds
one row for that choice rather than one per tried `k`.

__Step 35:__ The table again, with the wrapper rows in it.

In [ ]:
print(pd.DataFrame(results).T.to_string(formatters=RESULT_FORMAT, na_rep=""))

# <font color='#E8800A'>Feature selection III: Embedded methods</font> <a class="anchor" id="embedded"></a>
[Back to TOC](#toc)

An embedded method selects during model fitting. `Lasso` supplies
L1-penalised coefficients, while a tree supplies impurity-based importances.
A fixed penalty separates feature selection from hyperparameter search.

<div class="alert alert-block alert-info">

**A coefficient or an importance is a score, not a decision.** A
fitted `Lasso` gives every column a coefficient, and a fitted tree gives every
column an impurity importance. Neither says which columns to keep.
[`SelectFromModel`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html)
makes that decision: it fits the model, reads `coef_` or `feature_importances_`,
and keeps the columns whose absolute value reaches `threshold`.

The threshold therefore carries the decision, and each signal needs its own:

- **Lasso.** The L1 penalty itself sets weak coefficients to exactly zero, so
  `threshold=1e-8` keeps the columns the penalty left alive. The penalty
  strength `alpha` decides how many survive, and a penalty small enough to
  shrink nothing selects nothing.
- **Tree importance.** Importances sum to 1 and have no natural cut, so
  `threshold="median"` keeps the more important half by construction. That width
  is chosen in advance; it is not a finding about the data.

</div>

__Step 36:__ **Lasso embedded, two attempts.** The penalty strength decides what
survives, so measure the declared `alpha=0.01` and a much stronger `alpha=30`. Then average the stronger model's coefficients over the splits; the whisker is the spread across splits, and a grey bar marks a
column the penalty removed in most of them.

In [ ]:
measure("Lasso embedded", lambda split_number: SelectFromModel(
    Lasso(alpha=0.01, random_state=RANDOM_STATE), threshold=1e-8))
report("Lasso embedded")

measure("Lasso embedded, alpha=30", lambda split_number: SelectFromModel(
    Lasso(alpha=30, random_state=RANDOM_STATE), threshold=1e-8))
report("Lasso embedded, alpha=30")

In [ ]:
coefficients = per_split("Lasso embedded, alpha=30",
                         lambda fitted: fitted.estimator_.coef_)
kept_share = per_split("Lasso embedded, alpha=30",
                       lambda fitted: fitted.get_support()).astype(float).mean()
lasso_order = coefficients.mean().abs().sort_values().index
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(lasso_order, coefficients.mean()[lasso_order],
        xerr=coefficients.std()[lasso_order],
        color=np.where(kept_share[lasso_order] > 0.5, PLOT_BLUE, "0.75"))
ax.axvline(0, color="0.3", linewidth=0.8)
ax.set(title=f"Lasso coefficients at alpha=30, mean of {len(coefficients)} splits:"
             f" {int((kept_share > 0.5).sum())} of {len(lasso_order)} kept in most",
       xlabel="coefficient, EUR per scaled unit (whisker: spread across splits)",
       ylabel="")
fig.tight_layout()
plt.show()

The declared penalty selects nothing: at `alpha=0.01` every
coefficient survives in every split, and the row is identical to keeping every
column. The stronger penalty is a different selector. At `alpha=30` it keeps 16.3
columns and reaches 3,084.63 EUR, a gap of -2.36 +- 1.81 against all features,
better on 14 of the 20 splits and inside the noise.

The averaged coefficients show what it removes. `paintQuality%` and
`previousOwners`, the two weakest columns in the relevance table, go to zero in
most splits, and so do `tax` and several fuel and transmission levels. The
survivors are large and steady: `engineSize` and `year` carry about 40,000 and
32,000 EUR across their scaled ranges, and `mpg` about -19,000. Choosing
`alpha=30` is itself a search on these folds, so this row carries the same
optimism as any other setting chosen here.

__Step 37:__ **Tree importance embedded.** Measure a tree with
`threshold="median"`, then average its importances the same way. The dashed line
is the median cut, averaged over the splits.

In [ ]:
measure("DT importance embedded", lambda split_number: SelectFromModel(
    DecisionTreeRegressor(random_state=RANDOM_STATE), threshold="median"))
report("DT importance embedded")

In [ ]:
importances = per_split("DT importance embedded",
                        lambda fitted: fitted.estimator_.feature_importances_)
kept_share = per_split("DT importance embedded",
                       lambda fitted: fitted.get_support()).astype(float).mean()
cut = np.mean([fitted.threshold_ for fitted in fitted_by_split["DT importance embedded"]])
tree_order = importances.mean().sort_values().index
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(tree_order, importances.mean()[tree_order], xerr=importances.std()[tree_order],
        color=np.where(kept_share[tree_order] > 0.5, PLOT_ORANGE, "0.75"))
ax.axvline(cut, color="0.3", linestyle="--", linewidth=1)
ax.set(title=f"Tree importances, mean of {len(importances)} splits",
       xlabel="impurity importance\n(whisker: spread across splits; dashed: mean median cut)",
       ylabel="")
fig.tight_layout()
plt.show()
print(f"mean median cut {cut:.4f}; kept in most splits:"
      f" {int((kept_share > 0.5).sum())} of {len(tree_order)}")

Averaged over the splits, `transmission_Manual`, `engineSize` and
`year` take about three quarters of the importance, and the mean median cut,
0.0043, falls in the flat tail below them. That cut keeps `paintQuality%` and
`previousOwners` in every split, although their |Pearson r| with price are 0.0345
and 0.0076 in the relevance table: a near-continuous column offers a tree many
places to split, and impurity reduction rewards that even when the column says
little about price.

The result is the worst row on the board: 12.6 columns at 3,279.44 EUR, +192.44
+- 12.96 against all features and worse on every split.

__Step 38:__ The table again, with the embedded rows in it.

In [ ]:
print(pd.DataFrame(results).T.to_string(formatters=RESULT_FORMAT, na_rep=""))

# <font color='#E8800A'>Combining strategies</font> <a class="anchor" id="combined"></a>
[Back to TOC](#toc)

Selectors can run **in sequence**, each narrowing the next one's
input, or in parallel as a **vote**. Both combinations below use the same three
techniques with the same settings, so the only thing
that differs between them is the rule that combines the techniques.

A combination is what this notebook exports, because the algorithm is not chosen
yet. Each family reads features through its own instrument: a filter through one
column at a time, a wrapper through one fitted estimator, an embedded method
through the model it is part of. A set that two or three of those readings agree
on leans less on any one of them, and the weeks ahead fit other algorithms on
it. That is the argument for combining
rather than a result this notebook measures: measuring it would take a second
algorithm, which is a later week's work.

So the question here is not which technique wins. It is which rule for combining
them to export.

<div class="alert alert-block alert-info">

A **sequence** forms a funnel: each technique sees only the columns
the previous one kept, so the later ones fit on fewer columns, but an early
rejection is final and can erase a useful joint effect.

A **vote** fits every technique on the same frame and keeps the columns with
enough support. One technique cannot decide the result, but shared blind spots
survive the vote, and every technique still pays its full fitting cost.

</div>

__Step 39:__ Write both combinations as selectors. Each records, in `decisions_`,
which technique kept each column and whether the combination kept it, so the fitted copies from every split can be read together rather than one at a time. Both
can also describe the configuration they were built from, because the strategy
this notebook ends up selecting has to reach the log in a form a later week can
rebuild.

In [ ]:
def describe(selector):
    """The constructor call that built a selector, as text for the log.

    A scikit-learn estimator keeps its parameters as plain attributes and marks
    everything it learned with a trailing underscore, so dropping those leaves
    the configuration it was handed.
    """
    def literal(value):
        if hasattr(value, "get_params"):
            return describe(value)
        if callable(value):
            return value.__name__
        return repr(value)

    parameters = (selector.get_params(deep=False)
                  if hasattr(selector, "get_params") else vars(selector))
    arguments = ", ".join(f"{name}={literal(value)}"
                          for name, value in parameters.items()
                          if not name.endswith("_"))
    return f"{type(selector).__name__}({arguments})"


class Sequence:
    """Apply selectors one after another, each on what the previous one left.

    The mask is composed rather than recomputed: step two is fitted on the
    columns step one kept, and its support is written back into their positions.
    `decisions_` says, for every column, whether it was still held after each
    stage, so the counts can only shrink from one stage to the next.
    """

    def __init__(self, *stages):
        self.stages = stages

    def fit(self, frame, outcome):
        self.feature_names_in_ = frame.columns.to_numpy()
        support = np.ones(frame.shape[1], dtype=bool)
        decisions = {}
        for name, stage in self.stages:
            survivors = frame.loc[:, support]
            kept = stage.fit(survivors, outcome).get_support()
            support[np.flatnonzero(support)] = kept
            decisions[name] = support.copy()
        self.support_ = support
        self.decisions_ = pd.DataFrame(decisions, index=frame.columns)
        self.decisions_["kept"] = support
        return self

    def get_support(self):
        return self.support_

    def spec(self):
        """The stages in order, each as the constructor call that built it."""
        return {"kind": "sequence",
                "stages": [{"stage": name, "selector": describe(stage)}
                           for name, stage in self.stages]}


class Vote:
    """Keep the columns that at least `minimum` of the members select.

    Every member is fitted on the same training rows, so the vote is over
    methods rather than over rows, and a column kept by one member alone is not
    kept by the vote. `decisions_` records which member kept which column.
    """

    def __init__(self, members, minimum):
        self.members = members
        self.minimum = minimum

    def fit(self, frame, outcome):
        self.feature_names_in_ = frame.columns.to_numpy()
        self.decisions_ = pd.DataFrame(
            {name: member.fit(frame, outcome).get_support()
             for name, member in self.members.items()},
            index=frame.columns,
        )
        self.support_ = (self.decisions_.sum(axis=1) >= self.minimum).to_numpy()
        self.decisions_["kept"] = self.support_
        return self

    def get_support(self):
        return self.support_

    def spec(self):
        """The threshold and the members, each as the call that built it."""
        return {"kind": "vote", "minimum": self.minimum,
                "members": {name: describe(member)
                            for name, member in self.members.items()}}


def majority_table(name):
    """Keep counts over the twenty splits, for the columns kept in most of them.

    Each entry counts the splits in which that technique kept the column, and
    the last column counts the splits in which the combination kept it. A
    column the combination kept in half the splits or fewer is left out.
    """
    runs = fitted_by_split[name]
    counts = pd.concat([run.decisions_ for run in runs]).groupby(level=0).sum()
    shown = counts[counts["kept"] > len(runs) / 2].sort_values(
        list(counts.columns[::-1]), ascending=False)
    print(f"{len(shown)} of {len(counts)} columns kept in most of the"
          f" {len(runs)} splits")
    print((shown.astype(int).astype(str) + f"/{len(runs)}").to_string())

__Step 40:__ **The three shared techniques**, one per family: the correlation
filter drops every column that repeats one already kept and spends no relevance
budget, `RFE` narrows to the width the search above selected, and the
`Lasso` at `alpha=30` keeps the coefficients its penalty leaves alive. Every call builds fresh, unfitted copies, so nothing fitted in one
split reaches another. These settings were chosen on this board so that neither
combination is handicapped.

In [ ]:
def shared_techniques():
    """One technique per family, in funnel order: filter, wrapper, embedded."""
    return [
        ("filter", CorrelationFilter(None, redundancy=REDUNDANT)),
        ("wrapper", RFE(Ridge(random_state=RANDOM_STATE),
                        n_features_to_select=chosen_k)),
        ("embedded", SelectFromModel(Lasso(alpha=30, random_state=RANDOM_STATE),
                                     threshold=1e-8)),
    ]

__Step 41:__ **A sequence.** Run the three in that order, each on the columns the
previous one kept.

In [ ]:
measure("sequence: filter -> wrapper -> embedded",
        lambda split_number: Sequence(*shared_techniques()))
report("sequence: filter -> wrapper -> embedded")

__Step 42:__ Read the sequence over every split. The table lists only the
columns the sequence kept in most splits, and each entry counts the splits in
which that stage still held the column.

In [ ]:
majority_table("sequence: filter -> wrapper -> embedded")

The funnel keeps 14.6 columns and reaches 3,081.54 EUR, a gap of
-5.46 +- 1.86 against all features, better on 16 of the 20 splits: the lowest
error on the board, and the only selection row whose gain survives two
standard errors. Two fifths of the frame is gone and the model predicts better
without it.

The table shows how little each stage takes from the one before it.
Thirteen columns survive all three stages in every split, and `Brand_Ford` is the
only column any stage treats differently: the `Lasso` drops it in three splits
after the filter and the wrapper kept it in twenty. The early stages remove
nothing the later ones would have wanted.

__Step 43:__ **A vote.** Fit the same three on the full training matrix, and keep
a column when at least two of them keep it.

In [ ]:
measure("vote: 2 of 3 families",
        lambda split_number: Vote(dict(shared_techniques()), minimum=2))
report("vote: 2 of 3 families")

__Step 44:__ Read the vote the same way. Each entry now counts the splits in which
that technique, on its own, kept the column, and the last column counts the
splits in which at least two of them agreed.

In [ ]:
majority_table("vote: 2 of 3 families")

The vote keeps 20.9 columns and reaches 3,085.32 EUR, a gap of
-1.67 +- 0.94, better on 9 of the 20 splits and inside the noise. It keeps 6.3
more columns than the sequence on average, because the redundancy filter votes for
nearly every column, so any column one of the other two keeps is carried.

Their majority tables differ by seven columns, and the vote table shows which
techniques carried each one. `Brand_BMW` survives although the wrapper keeps it in
only **7 of 20** splits, because the filter and the `Lasso` both keep it in
twenty. `tax`, `fuelType_Other`, the two one-car levels and
`transmission_Automatic` go in the other direction: the `Lasso` removes them in
every split, and the filter and the wrapper outvote it. `Brand_VW` is the
seventh, kept by all three in most splits yet by the sequence in half of them or
fewer.

__Step 45:__ The table again, with the combinations in it: every technique and
every attempt.

In [ ]:
print(pd.DataFrame(results).T.to_string(formatters=RESULT_FORMAT, na_rep=""))

<div class="alert alert-block alert-success">

Three rows match the full matrix's error with fewer columns: the sequence
at 14.6 columns for -5.46 EUR, the `Lasso` at `alpha=30` at 16.3 for -2.36, and
the vote at 20.9 for -1.67. The sequence is the only one whose gain clears two
standard errors, and it is also the narrowest. The `RFE` row carries no verdict,
because its width was chosen on these splits.

Every row whose width was fixed in advance is worse, by 67 to 192 EUR. On this
frame a width chosen by a fitted model holds up and a width fixed in advance does
not.

</div>

# <font color='#E8800A'>So what is selection for?</font> <a class="anchor" id="synthesis"></a>
[Back to TOC](#toc)

Selection reduced the error measurably once, and by little; what it changed
most is the width of the model.

__Step 46:__ **Exercise.** Collect the rows every section above wrote into
`board`: mean MAE with its across-split standard error, and the mean number of
features. The plot beneath it is given.

In [ ]:
summary = pd.DataFrame({
    name: {
        # 1. mean feature count and mean MAE over the splits
        "n": ...,  # <-- CODE HERE
        "MAE": ...,  # <-- CODE HERE
        # 2. across-split sem: a raw standard deviation would describe
        #    the spread of the splits, not the precision of their mean
        "MAE sem": ...,  # <-- CODE HERE
    }
    for name, table in board.items()
}).T
# 3. order the board by MAE ascending, lowest first
summary = ...  # <-- CODE HERE

print(f"{'selector':40s} {'n':>5s} {'MAE (EUR)':>10s} {'sem':>6s}")
for name, row in summary.iterrows():
    # 4. one line per selector, aligned under the header above
    ...  # <-- CODE HERE

# Given: the table as a figure.
plt.figure(figsize=(7, 5))
plt.errorbar(summary["MAE"], np.arange(len(summary)),
             xerr=2 * summary["MAE sem"],
             fmt="o", color=PLOT_BLUE, capsize=4)
plt.yticks(np.arange(len(summary)), summary.index)
plt.gca().invert_yaxis()
plt.xlabel("held-out MAE, EUR  (bars are 2 x across-split SEM)")
plt.title("Feature-selection error across repeated holdouts")
plt.tight_layout()
plt.show()


In [ ]:
# Same budget of ten, a wrapper instead of a filter.
gap = paired_delta(board["RFE(k=10)"]["MAE"].to_numpy(),
                   board["SelectKBest(k=10)"]["MAE"].to_numpy())
print(f"wrapper instead of filter, k=10: dMAE {gap['dMAE']:+.2f} +- {gap['SEM']:.2f} EUR"
      f"   better on {gap['better']} splits -> {gap['verdict'].upper()}")

<div class="alert alert-block alert-success">

At ten columns, the wrapper
improves on the filter by **55.40 EUR ± 12.16** on **16 of 20** splits: the same
budget, spent by a fitted model instead of a ranking.

An MAE difference can be statistically clear and still be commercially trivial,
so the effect in euros is reported with its SEM and the retained width: 55.40
EUR is under 2% of the 3,087 EUR error.

</div>

One strategy is measurably better than all **24.6** features: the sequence, at
**-5.46 EUR +- 1.86** on 14.6 columns. That gain is under two parts in a thousand
of the 3,087 EUR error, while the fixed-budget rows cost from **+67.19 EUR** to
**+192.44 EUR**. What selection buys here is width.

Feature construction moves the error more than selection does. The ruled
transforms gain **59.53 EUR**, `km_per_year` gains **32.74**, and logging
`engineSize` costs **98.43**.

__Step 47:__ **The decisions this notebook exports.** Record both decisions in one
cell, then write the log once: the encoder and scaler combination, with the
scores of the sixteen combinations it was chosen from, and the selection
strategy. The candidates are the two combinations, for the reason the combining
section gives, and the one with the lower mean paired error gap to all features
wins, because a negative gap is an improvement.

In [ ]:
# 1. the encoder + scaler combination chosen at the start, as ONE decision,
#    with the scores of all sixteen combinations it was chosen from
recipe_log.record(
    f"all {len(categorical) + len(numeric)} feature columns",
    f"selected the {selected_encoding} + {selected_scaler} combination: {selected_encoding}"
    f" encoding and {selected_scaler} scaling, both fitted inside each training split",
    "lowest mean held-out MAE among the sixteen combinations of the four encoders"
    " and the four Week 3 scalers; Ridge is penalised and so not scale-invariant,"
    " which is the re-measurement the Week 3 'NO scaler' entry asks for",
    len(X),
    carries={
        "combination": f"{selected_encoding} + {selected_scaler}",
        "encoding": selected_encoding,
        "scaler": selected_scaler,
        "drop": "first" if selected_encoding == "one-hot" else None,
        "categorical": list(categorical),
        "numeric": list(numeric),
        "dropped": ["model"],
        "fitted_per_split": True,
        "evaluation": "20 repeated 80/20 holdouts",
        "metric": "test MAE",
        "candidates": {
            f"{encoding_name} + {scaler_name}": {
                metric: float(value) for metric, value in row.items()
            }
            for (encoding_name, scaler_name), row in encoding_board.iterrows()
        },
    },
)

# 2. the selection strategy: the better of the two combinations
candidates = {
    name: board[name]["MAE"].to_numpy() - board["all features"]["MAE"].to_numpy()
    for name in ("sequence: filter -> wrapper -> embedded", "vote: 2 of 3 families")
}
best = min(candidates, key=lambda name: candidates[name].mean())

recipe_log.record(
    f"{len(first_names)} encoded columns",
    f"select features with the {best} strategy, refitted inside each training split",
    "smaller paired MAE gap to the all-features arm than the other combination;"
    " a combination is used so that no single technique decides the retained set",
    len(X),
    carries={
        "strategy": selector_examples[best].spec(),
        "fitted_per_split": True,
        "evaluation": "20 repeated 80/20 holdouts",
        "metric": "paired dMAE against the all-features arm, in euros",
        "candidates": {name: {"dMAE": float(gap.mean()),
                              "n": float(board[name]["n"].mean())}
                       for name, gap in candidates.items()},
        "applied": "from Week 5 onward",
    },
)

# 3. one write, with both of this week's decisions in it
recipe_log.to_json("../../logs/week_04_feature_work_regression_log.json")

print(f"logged: {selected_encoding} + {selected_scaler}, then {best}")
print(f"{len(recipe_log)} decisions written to"
      " ../../logs/week_04_feature_work_regression_log.json")

`carries["strategy"]` holds each stage as the call that built it, so the next
notebook rebuilds the sequence and refits it inside its own splits. A list of the
columns kept here would carry a decision made on these 3,200 training rows into
rows it never saw.

The sequence won, -5.46 EUR against -1.67 for the vote, and it carries 14.6
columns instead of 20.9. Its gain is the one that survives two standard errors,
so it is the rule the log records.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. **Declare the regime before reading results.** The comparison,
   split, metric, and reference floor must be fixed in advance.
2. **Choose transforms per column.** Adding `engineSize` turns a 59.53 EUR gain
   into a 38.24 EUR loss, and logging `km_per_year` gives back 22.31 of its
   32.74 EUR gain.
3. **Reduction keeps variance, not signal.** Eight PCA components retain 80% of
   training variance but cost 2,279.52 EUR on every split.
4. **Check what a filter measures.** Variance can rank units or category
   frequency by accident; relevance and redundancy require different indices.
5. **Price selection in the target's units.** Compare methods at a matched width,
   then report the paired MAE effect, its SEM and the retained width.
6. **Export a combination, not the best single row.** The algorithm is not
   chosen yet, and each family reads features through its own instrument, so a
   set two of three families keep depends less on any one of them.
7. **Carry the strategy, not the columns.** The retained set belongs to the rows
   it was fitted on. What the log exports is the rule that produced it, so the
   next notebook refits that rule inside its own splits.

### Apply this method

For each transform, record a paired model delta and test derived features beside
any transformations of their parents. Compare a filter with a wrapper at a
matched budget, and treat a width chosen on validation splits as a choice, not a
score. Keep every target-aware fit inside its training split. If the target is transformed, state what the resulting error measures and
retain an error in the original units for comparison.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Guyon, I. & Elisseeff, A. (2003). An introduction to variable and feature selection. *JMLR*, 3, 1157-1182.
- Kuhn, M. & Johnson, K. (2019). *Feature Engineering and Selection: A Practical Approach for Predictive Models*. CRC Press, § 1.4.
- Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *JRSS B*, 58(1), 267-288.
- Jolliffe, I.T. & Cadima, J. (2016). Principal component analysis: a review and recent developments. *Phil. Trans. R. Soc. A*, 374(2065).
- scikit-learn User Guide, [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html) · [`RFECV`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFECV.html) · [`SequentialFeatureSelector`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) · [`SelectFromModel`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html) · [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) · [`VarianceThreshold`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html).
- numpy documentation, [`numpy.linalg.svd`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html).